# Lean from Python

[LeanInteract](https://github.com/augustepoiroux/LeanInteract) sends Lean code to a Lean REPL
and returns the result as Python objects: errors, warnings, and the open goals of each `sorry`.
You write Lean as strings in Python cells, and Python decides what to try next.

The REPL was built for this booth's Lean version when the image was built, so the next cell
starts in a few seconds.

In [ ]:
from pathlib import Path
from lean_interact import Command, LeanREPLConfig, LeanServer, ProofStep

# The toolchain this folder pins (lean-toolchain), e.g. v4.35.0.
version = Path("lean-toolchain").read_text().strip().split(":")[-1]
server = LeanServer(LeanREPLConfig(lean_version=version))
print("Lean", version, "ready")

## 1. Check a theorem

A reply with no errors means Lean accepted the proof.

In [ ]:
reply = server.run(Command(cmd="theorem add_zero' (n : Nat) : n + 0 = n := by simp"))
print("valid:", reply.lean_code_is_valid(allow_sorry=False))

## 2. A false theorem is rejected

Lean does not take your word for it. The error says which goal it could not close.

In [ ]:
reply = server.run(Command(cmd="theorem wrong : 2 + 2 = 5 := by decide"))
print("valid:", reply.lean_code_is_valid(allow_sorry=False))
for e in reply.get_errors():
    print(f"line {e.start_pos.line}:", e.data)

## 3. Build on earlier results

Each reply has an `env` number. Pass it back to keep using what was defined before, the same way
later lines in a Lean file see earlier ones.

In [ ]:
defs = server.run(Command(cmd='''
def oddSum : Nat → Nat
  | 0     => 0
  | n + 1 => oddSum n + (2 * n + 1)
'''))
for n in [3, 10, 25]:
    reply = server.run(Command(cmd=f"example : oddSum {n} = {n} * {n} := by decide", env=defs.env))
    print(f"oddSum {n} = {n}²:", reply.lean_code_is_valid(allow_sorry=False))

## 4. Prove step by step

Leave the proof as `sorry` and Lean returns the goal still to prove, with a `proof_state` number.
Send one tactic at a time with `ProofStep` and watch the goal change. This is the Infoview of
VS Code, as Python data.

In [ ]:
reply = server.run(Command(cmd="theorem swap (a b : Nat) : a + b + 0 = b + a := by sorry"))
goal = reply.sorries[0]
print("start:", goal.goal, sep="\n")

state = goal.proof_state
for tactic in ["rw [Nat.add_zero]", "rw [Nat.add_comm]"]:
    step = server.run(ProofStep(tactic=tactic, proof_state=state))
    state = step.proof_state
    print(f"\nafter {tactic}:", "\n".join(step.goals) or "no goals", sep="\n")
print("\nproof:", step.proof_status)

## 5. Search for proofs

Try several tactics on several statements and keep what Lean accepts. This is the loop behind AI
theorem proving, minus the AI: something proposes, Lean checks. `proof_search.py` is the same
code as a script (`just search`).

In [ ]:
import proof_search

for statement in proof_search.STATEMENTS:
    tactic = proof_search.search(server, statement)
    print(f"{'✓' if tactic else '✗'} {statement:<42} {tactic or 'no tactic worked'}")